# Session 5: XGBoost and the Single Test Evaluation
**Emission-trajectory ML project. Dr. Khawar Naeem, QTTSC, Qatar University.**

The last modeling session. Two things happen here, in this order and no other:

1. **Validation tuning.** A small, pre-frozen XGBoost grid is tuned on the
   validation years (targets 2015-2018) only, in both parameterizations
   (predict the level, and predict the delta). Nothing about the test set is
   touched.
2. **The single test evaluation.** After the configuration is frozen in
   writing and the test pre-registration is filled in, ALL eight models are
   scored ONCE on the test years (targets 2019-2023), same-rows rule. After
   that run, nothing is retuned on any split. Session 6 interprets; it does
   not re-fit.

**Leakage boundary.** Features for predicting year t+1 use information through
year t only (enforced already in `src/build_features.py`). The test years are
never used for model selection. The `RUN_FINAL_TEST` flag below stays `False`
until the freeze and the pre-registration are done.

**This session is not about making XGBoost win.** It is about finding out
honestly whether its nonlinear flexibility generalizes better than persistence
through the difficult 2019-2023 period, which includes the 2020 COVID break.

Run in the reproducible Anaconda environment (Python 3.12, scikit-learn 1.5.1,
xgboost 3.3.0), per `requirements.txt`.

## 0. Pre-registration, validation phase (fill BEFORE running any model cell)

Commit these in writing first, so the output becomes a test of understanding,
not a number to scroll past. Uncertainty is acceptable; reasoning is the point.

1. Will XGBoost (either parameterization) beat persistence on **validation**
   MAE, where Ridge and HistGB both failed? Why or why not?
   > *your answer here*
2. Level versus delta for XGBoost: which do you expect to tune better, given
   what happened to `hgb_level` in Session 4 (skill -1.79, trees cannot
   extrapolate a rising level)?
   > *your answer here*
3. Deeper trees (`max_depth`) and a smaller `learning_rate`: which way does
   each push overfitting, and which do you expect early stopping to prefer?
   > *your answer here*

## 1. Theory: what XGBoost adds to gradient boosting

(Formal, numbered version in `math/main.pdf` Section 4; this is the intuition.)

**Boosting recap.** An additive model $F_M(x)=\sum_{m=1}^{M}\gamma_m h_m(x)$
built one tree at a time, each new tree $h_m$ fitted to the errors the ensemble
still makes. In gradient boosting each tree fits the negative gradient
(pseudo-residual) of the loss; for squared error that is just the ordinary
residual $y_i-F_{m-1}(x_i)$.

**What XGBoost adds.**
- **A regularized objective.** It penalizes model complexity directly:
  $\Omega(h)=\gamma T+\tfrac{1}{2}\lambda\lVert w\rVert^2$, where $T$ is the
  number of leaves and $w$ the leaf values. This discourages large, overfit
  trees before they are even grown.
- **A second-order (Newton) step.** Instead of using only the gradient $g_i$,
  it also uses the second derivative (Hessian) $h_i$ of the loss. Minimizing
  the second-order Taylor expansion gives a closed-form optimal leaf weight
  $w^*_j=-\tfrac{\sum_{i\in j} g_i}{\sum_{i\in j} h_i+\lambda}$ and an exact
  split-gain formula, so splits are chosen to reduce the true regularized loss,
  not just the residual sum of squares.

**The knobs, and what they trade.**
- `learning_rate` (shrinkage) x `n_estimators`: each tree's contribution is
  scaled by the learning rate, so a smaller rate needs more trees to reach the
  same fit. Small rate + many trees generalizes better but costs compute;
  they trade off directly.
- `max_depth`: deeper trees capture more interactions but overfit faster.
- `subsample`, `colsample_bytree`: fit each tree on a random fraction of rows
  and columns, which decorrelates the trees and curbs overfitting.
- **Early stopping**: keep adding trees only while a watched validation metric
  improves; stop when it stalls. The stopping point (`best_iteration`) is the
  frozen number of trees. This is tuning on the VALIDATION years, which is
  allowed. Choosing anything by looking at the TEST years is peeking, which is
  not: the test set must be touched once, after every choice is frozen.

**Why trees still need the delta trick here.** A tree predicts piecewise
constants bounded by the training targets. A country whose emissions rise above
anything seen in training (a growing giant) cannot be predicted in the level
parameterization; the tree saturates and under-predicts, exactly the
`hgb_level` failure from Session 4. Predicting the delta and adding it to the
known current level sidesteps this. XGBoost is tested in both parameterizations
for the same reason.

## 2. Load the modeling table, features, and splits

The same table, the same feature lists, and the same chronological splits as
every prior session. No model in this project differs in its data, only in its
algorithm.

In [ ]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
import xgboost as xgb
import evaluate

SEED = 42

df = pd.read_csv("../data/processed/modeling_table.csv")

# Identical feature lists to Session 4. 'year' is deliberately excluded (a tree
# would memorize era boundaries it cannot extrapolate).
CORE = ["co2", "co2_lag1", "co2_lag3", "co2_lag5", "co2_lag10",
        "co2_roll5_mean", "co2_roll10_mean", "co2_slope5",
        "co2_per_capita", "share_global_co2",
        "population", "pop_growth5_pct", "cement_co2", "flaring_co2"]
NULLABLE = ["primary_energy_consumption", "energy_growth1_pct", "energy_per_capita"]
FEATURES = CORE + NULLABLE

train = df[df["split"] == "train"].copy()   # targets <= 2014
val   = df[df["split"] == "val"].copy()     # targets 2015-2018
test  = df[df["split"] == "test"].copy()    # targets 2019-2023 (LOCKED until the freeze)
prov  = df[df["split"] == "provisional_2024"].copy()

print(f"xgboost {xgb.__version__}")
print(f"train {len(train)}  val {len(val)}  test {len(test)}  provisional {len(prov)}  |  {len(FEATURES)} features")
print("target years -> train <= 2014, val 2015-2018, test 2019-2023, provisional 2024")

## 3. Validation tuning: a small, pre-frozen XGBoost grid

The grid is deliberately small and frozen BEFORE its results are seen: two
`max_depth` values x two `learning_rate` values = four configurations, each
with a high tree ceiling and early stopping to choose the useful number of
trees on the validation years. `subsample` and `colsample_bytree` are fixed,
the seed is fixed. We do this for both parameterizations (level and delta):
eight fits in total. Selection is by validation MAE on the reconstructed
LEVEL, so every model stays comparable on the same quantity.

The question is generalization through time, not leaderboard optimization on
the 2015-2018 window, so the grid stays tiny.

In [ ]:
# Frozen search space (committed before seeing results):
MAX_DEPTHS    = (3, 6)
LEARNING_RATES = (0.05, 0.10)
N_TREES_CEILING = 2000
EARLY_STOP = 50
SUBSAMPLE = 0.8
COLSAMPLE = 0.8
REG_LAMBDA = 1.0

def tune_xgb(target_col):
    """Fit each grid config on train, early-stop on val, return a list of
    (params, best_n_trees, val_level_MAE). Predictions are always scored on the
    reconstructed LEVEL so level and delta models are comparable."""
    results = []
    for md_ in MAX_DEPTHS:
        for lr in LEARNING_RATES:
            model = xgb.XGBRegressor(
                n_estimators=N_TREES_CEILING, learning_rate=lr, max_depth=md_,
                subsample=SUBSAMPLE, colsample_bytree=COLSAMPLE,
                reg_lambda=REG_LAMBDA, random_state=SEED, n_jobs=-1,
                early_stopping_rounds=EARLY_STOP, eval_metric="mae",
            )
            model.fit(train[FEATURES], train[target_col],
                      eval_set=[(val[FEATURES], val[target_col])], verbose=False)
            best_n = model.best_iteration + 1
            raw = model.predict(val[FEATURES])
            level_pred = raw if target_col == "target_co2_next" else val["co2"].to_numpy() + raw
            mae = evaluate.mae(val["target_co2_next"], level_pred)
            results.append({"max_depth": md_, "learning_rate": lr,
                             "best_n_trees": best_n, "val_MAE": mae})
    return pd.DataFrame(results).sort_values("val_MAE").reset_index(drop=True)

xgb_level_grid = tune_xgb("target_co2_next")
xgb_delta_grid = tune_xgb("target_delta")
print("XGBoost LEVEL grid (val MAE, best first):"); print(xgb_level_grid.to_string(index=False))
print(); print("XGBoost DELTA grid (val MAE, best first):"); print(xgb_delta_grid.to_string(index=False))

## 4. Freeze the XGBoost configuration (your decision)

Read the two grids above, then commit your final choices below. This is a
Khawar decision, like the Ridge alpha freezes in Session 4. Fill in each
`...`. After this cell, the hyperparameters are frozen; the test set is next.

In [ ]:
# Fill these from the grids above, then run. Ellipsis (...) will error until set,
# which is the intended 'stop and decide' guard.
XGB_LEVEL = {"max_depth": ..., "learning_rate": ..., "n_trees": ...}   # from xgb_level_grid
XGB_DELTA = {"max_depth": ..., "learning_rate": ..., "n_trees": ...}   # from xgb_delta_grid

assert Ellipsis not in XGB_LEVEL.values(), "freeze XGB_LEVEL from the level grid first"
assert Ellipsis not in XGB_DELTA.values(), "freeze XGB_DELTA from the delta grid first"
print("frozen level:", XGB_LEVEL)
print("frozen delta:", XGB_DELTA)

## 5. Test pre-registration (fill BEFORE unlocking the test run)

Record your expectations now, before a single test number is seen. Also write
the two short 'XGBoost wins' and 'XGBoost loses' post drafts in the private,
gitignored `linkedin_drafts/` folder before running the test (pre-registration
in public).

1. Rank all eight models by expected TEST MAE (best to worst): persistence,
   linear_trend, ridge_level, ridge_delta, hgb_level, hgb_delta, xgb_level,
   xgb_delta.
   > *your ranking*
2. Will ANY model achieve positive skill against persistence on the test years?
   > *your answer*
3. Will XGBoost level or delta do better on test, and will validation gains
   survive 2019-2023?
   > *your answer*
4. The 2020 COVID year: which models do you expect to miss it worst, and why?
   > *your answer*

In [ ]:
# Stays False until the freeze (cell above) and the pre-registration are done.
# Set to True for the SINGLE test run, then never retune anything.
RUN_FINAL_TEST = False

## 6. The single test evaluation

Once frozen: refit every ML model on TRAIN + VALIDATION (all target years
through 2018) using the frozen hyperparameters and the frozen number of trees
(no early stopping on the refit, the tree count is already chosen). The two
baselines have no parameters, so they are not refit. Then score all eight
models ONCE on the test rows (targets 2019-2023), same-rows rule, keeping 2020.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor

# Ridge alphas frozen in Session 4; HistGB settings unchanged from Session 4.
RIDGE_ALPHA = 0.1

def _level(pred, base_co2, target_col):
    return pred if target_col == "target_co2_next" else base_co2 + pred

if RUN_FINAL_TEST:
    fit_df = pd.concat([train, val], ignore_index=True)   # all data through target-year 2018
    X_fit = fit_df[FEATURES]
    Xt = test[FEATURES]; base_t = test["co2"].to_numpy()

    def ridge_pred(target_col):
        pipe = Pipeline([("impute", SimpleImputer(strategy="median")),
                         ("scale", StandardScaler()),
                         ("model", Ridge(alpha=RIDGE_ALPHA))])
        pipe.fit(X_fit, fit_df[target_col])
        return _level(pipe.predict(Xt), base_t, target_col)

    def hgb_pred(target_col):
        m = HistGradientBoostingRegressor(max_depth=None, max_iter=300,
                                          learning_rate=0.05, early_stopping=False,
                                          random_state=SEED)
        m.fit(X_fit, fit_df[target_col])
        return _level(m.predict(Xt), base_t, target_col)

    def xgb_pred(target_col, cfg):
        m = xgb.XGBRegressor(n_estimators=cfg["n_trees"], learning_rate=cfg["learning_rate"],
                             max_depth=cfg["max_depth"], subsample=SUBSAMPLE,
                             colsample_bytree=COLSAMPLE, reg_lambda=REG_LAMBDA,
                             random_state=SEED, n_jobs=-1)
        m.fit(X_fit, fit_df[target_col])
        return _level(m.predict(Xt), base_t, target_col)

    test["pred_persistence"] = test["co2"]
    test["pred_linear_trend"] = test["co2"] + test["co2_slope5"]
    test["pred_ridge_level"] = ridge_pred("target_co2_next")
    test["pred_ridge_delta"] = ridge_pred("target_delta")
    test["pred_hgb_level"]   = hgb_pred("target_co2_next")
    test["pred_hgb_delta"]   = hgb_pred("target_delta")
    test["pred_xgb_level"]   = xgb_pred("target_co2_next", XGB_LEVEL)
    test["pred_xgb_delta"]   = xgb_pred("target_delta", XGB_DELTA)

    preds = {"persistence": "pred_persistence", "linear_trend": "pred_linear_trend",
             "ridge_level": "pred_ridge_level", "ridge_delta": "pred_ridge_delta",
             "hgb_level": "pred_hgb_level", "hgb_delta": "pred_hgb_delta",
             "xgb_level": "pred_xgb_level", "xgb_delta": "pred_xgb_delta"}
    test_table = evaluate.comparison_table(test, preds, persistence_col="pred_persistence")
    test_table.insert(1, "split", "test")
    display(test_table.sort_values("MAE").round(3))
else:
    print("TEST LOCKED. Freeze the config, fill the pre-registration, then set RUN_FINAL_TEST = True.")

## 7. Provisional 2024 table (separate, clearly labeled)

The newest OWID release year is provisional and revisable, so 2024 is reported
in its own labeled table, never mixed into the headline test result.

In [ ]:
if RUN_FINAL_TEST:
    Xp = prov[FEATURES]; base_p = prov["co2"].to_numpy()
    prov["pred_persistence"] = prov["co2"]
    prov["pred_linear_trend"] = prov["co2"] + prov["co2_slope5"]
    prov["pred_ridge_level"] = _level(Pipeline([("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()), ("model", Ridge(alpha=RIDGE_ALPHA))]).fit(
        pd.concat([train, val])[FEATURES], pd.concat([train, val])["target_co2_next"]).predict(Xp), base_p, "target_co2_next")
    # (Extend with the other models as needed; this table is an appendix, not the headline.)
    prov_table = evaluate.comparison_table(
        prov, {"persistence": "pred_persistence", "linear_trend": "pred_linear_trend",
               "ridge_level": "pred_ridge_level"}, persistence_col="pred_persistence")
    prov_table.insert(1, "split", "provisional_2024")
    display(prov_table.round(3))
else:
    print("Provisional 2024 table also locked behind RUN_FINAL_TEST.")

## 8. Significance layer (optional, computed once on the final test table)

A point ranking does not say whether a gap is real. `evaluate.cluster_bootstrap
_skill` puts a country-clustered 95% confidence interval and a bootstrap
p-value on each model's skill against persistence (formalized in `math/main.pdf`
Section 6). Because a country's own years are correlated, it resamples whole
countries, not rows. Run this ONCE, on the frozen test predictions, and report
the interval beside every point estimate. Which test to adopt and how to read
it remain your calls.

In [ ]:
if RUN_FINAL_TEST:
    for name, col in [("linear_trend", "pred_linear_trend"),
                       ("ridge_delta", "pred_ridge_delta"),
                       ("hgb_delta", "pred_hgb_delta"),
                       ("xgb_level", "pred_xgb_level"),
                       ("xgb_delta", "pred_xgb_delta")]:
        r = evaluate.cluster_bootstrap_skill(test, col, n_boot=5000, seed=0)
        verdict = "indistinguishable from persistence" if r["ci_low"] <= 0 <= r["ci_high"] else "DISTINGUISHABLE"
        print(f"{name:<13} skill {r['skill']:+.3f}  95% CI [{r['ci_low']:+.3f}, {r['ci_high']:+.3f}]  p={r['p_value']:.3f}  {verdict}")
else:
    print("Significance layer runs after the single test evaluation.")

## 9. Reconcile and interpret

Answer in your own words, after the single test run:

1. Score your test pre-registration. What did the biggest miss misunderstand?
   > *...*
2. Did any model earn positive, and statistically distinguishable, skill
   against persistence on test? State the honest one-sentence headline the
   README should carry.
   > *...*
3. Did validation rankings survive into the test years, or did they reshuffle?
   What does that say about tuning on one window?
   > *...*
4. The 2020 COVID year: which models failed there, and is that a fault or an
   expected limit of extrapolation from history?
   > *...*

## Check questions (close Session 5 by answering these to Claude)

1. Why do `n_estimators` and `learning_rate` trade off?
2. What is the difference between validation-based early stopping and test-set
   peeking?
3. If XGBoost beats Ridge on validation but loses on test, what does that
   suggest?

**End-of-session protocol:** run `src/train.py` for a clean reproduction, run
`python src/evaluate.py` (self-test), commit the notebook + results +
`src/train.py`, update `CLAUDE.md` and `AGENTS.md`, extend `math/` with the
XGBoost derivation, update the README with confirmed findings only, and record
Session 6 (error analysis) as the exact next action.